# Breast Cancer Classification

This notebook trains a supervised classification model on the breast cancer dataset.
It imports the pipeline definition from `model.py` (which the autoresearch agent edits)
and writes metrics to `metrics.json` for the wrapper script to extract.

**This notebook is read-only context for autoresearch.** Edit `model.py` to change the model.

In [ ]:
import json
import os

import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

from model import make_pipeline, TEST_SIZE, RANDOM_STATE

## Load Data

In [ ]:
df = pd.read_csv(os.path.join("data", "breast_cancer.csv"))
X = df.drop(columns=["target"])
y = df["target"]

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print(f"Training samples: {len(X_train)}")
print(f"Validation samples: {len(X_val)}")
print(f"Features: {X.shape[1]}")

## Train

In [ ]:
pipeline = make_pipeline()
pipeline.fit(X_train, y_train)

print(f"Pipeline steps: {[name for name, _ in pipeline.steps]}")

## Evaluate

In [ ]:
y_train_pred = pipeline.predict(X_train)
y_val_pred = pipeline.predict(X_val)

train_accuracy = accuracy_score(y_train, y_train_pred)
val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Train accuracy: {train_accuracy:.6f}")
print(f"Val accuracy:   {val_accuracy:.6f}")
print()
print("Validation classification report:")
print(classification_report(y_val, y_val_pred, target_names=["malignant", "benign"]))

## Write Metrics

In [ ]:
# Count features selected by the pipeline
feature_selector = pipeline.named_steps.get("feature_selection")
if feature_selector is not None and hasattr(feature_selector, "get_support"):
    num_features = int(feature_selector.get_support().sum())
else:
    num_features = X.shape[1]

metrics = {
    "val_accuracy": float(val_accuracy),
    "train_accuracy": float(train_accuracy),
    "num_features": num_features,
}

with open("metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Metrics written to metrics.json: {metrics}")